In [1]:
# ============================================================
# 03b_hyperparameter_tuning.ipynb
# Random Forest Hyperparameter Tuning
# Breast Tumor Classification Project
# ============================================================

In [1]:
# ============================================================
# 1. Import Libraries
# ============================================================

import sys
from pathlib import Path

import pandas as pd
import joblib

# Add project root to Python path
sys.path.append(str(Path().resolve().parent))


In [4]:
# ============================================================
# 2. Import Custom Modules
# ============================================================

from src.config import (
    PROCESSED_DATA_FILE,
    MODEL_FILE,
    PARAM_GRID
)

from src.split import stratified_split

from src.tune import perform_grid_search


In [5]:
# ============================================================
# 3. Load Processed Dataset
# ============================================================

df = pd.read_csv(
    PROCESSED_DATA_FILE
)

df.head()


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst,Diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,1
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,1
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,1
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,1
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,1


In [6]:
# ============================================================
# 4. Split Features and Target
# ============================================================

X = df.drop(columns=['Diagnosis'])

y = df['Diagnosis']

In [7]:
# ============================================================
# 5. Perform Stratified Split
# ============================================================

(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test
) = stratified_split(X, y)

In [9]:
# ============================================================
# 6. Display Parameter Grid
# ============================================================

parameter_grid = PARAM_GRID

parameter_grid

{'n_estimators': [200, 500, 800],
 'max_depth': [None, 5, 10, 20],
 'min_samples_split': [2, 5, 10],
 'min_samples_leaf': [1, 2, 4],
 'max_features': ['sqrt', 0.5, 1.0],
 'class_weight': [None, 'balanced']}

In [10]:
# ============================================================
# 7. Run GridSearchCV
# ============================================================

grid_search = perform_grid_search(
    X_train,
    y_train
)

print("Grid search completed.")

Fitting 5 folds for each of 648 candidates, totalling 3240 fits
Grid search completed.


In [11]:
# ============================================================
# 8. Best Parameters
# ============================================================

print("Best Parameters:\n")

print(grid_search.best_params_)

Best Parameters:

{'class_weight': 'balanced', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}


In [12]:
# ============================================================
# 9. Best ROC-AUC Score
# ============================================================

print(
    f"Best ROC-AUC Score: "
    f"{grid_search.best_score_:.4f}"
)

Best ROC-AUC Score: 0.9891


In [13]:
# ============================================================
# 10. Best Model
# ============================================================

best_model = grid_search.best_estimator_

print(best_model)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)


In [14]:
# ============================================================
# 11. Save Best Model
# ============================================================

joblib.dump(
    best_model,
    MODEL_FILE
)

print(f"Best model saved at: {MODEL_FILE}")

Best model saved at: D:\Personal\Vired\Capstone Project\main\artifacts\models\rf_model.joblib


In [16]:
# ============================================================
# 12. Save Tuning Results
# ============================================================

tuning_results = pd.DataFrame(
    grid_search.cv_results_
)

tuning_results_path = (
    "../artifacts/metrics/tuning_results.csv"
)

tuning_results.to_csv(
    tuning_results_path,
    index=False
)

print(
    f"Tuning results saved at: "
    f"{tuning_results_path}"
)

Tuning results saved at: ../artifacts/metrics/tuning_results.csv


In [17]:
# ============================================================
# 13. Top 10 Configurations
# ============================================================

top_results = (
    tuning_results[
        [
            'params',
            'mean_test_score',
            'rank_test_score'
        ]
    ]
    .sort_values(
        by='rank_test_score'
    )
    .head(10)
)

top_results

,params,mean_test_score,rank_test_score
486,"{'class_weight': 'balanced', 'max_depth': 10, ...",0.989136,1
324,"{'class_weight': 'balanced', 'max_depth': None...",0.989136,1
567,"{'class_weight': 'balanced', 'max_depth': 20, ...",0.989136,1
327,"{'class_weight': 'balanced', 'max_depth': None...",0.988529,4
570,"{'class_weight': 'balanced', 'max_depth': 20, ...",0.988529,4
489,"{'class_weight': 'balanced', 'max_depth': 10, ...",0.988529,4
325,"{'class_weight': 'balanced', 'max_depth': None...",0.988322,7
568,"{'class_weight': 'balanced', 'max_depth': 20, ...",0.988322,7
487,"{'class_weight': 'balanced', 'max_depth': 10, ...",0.988322,7
326,"{'class_weight': 'balanced', 'max_depth': None...",0.988310,10


# Hyperparameter Tuning Summary

## Tuning Workflow

1. The processed dataset was loaded successfully and split into:
   - Training Set
   - Validation Set
   - Test Set

2. Stratified splitting was used to preserve class balance across all datasets.

3. `GridSearchCV` was used to evaluate multiple Random Forest hyperparameter combinations using 5-fold cross-validation.

4. The following hyperparameters were tuned:
   - `n_estimators`
   - `max_depth`
   - `min_samples_split`
   - `min_samples_leaf`
   - `max_features`
   - `class_weight`

5. ROC-AUC was selected as the primary scoring metric because:
   - it is threshold independent,
   - robust for binary classification,
   - and suitable for slightly imbalanced datasets.

---

# Best Hyperparameters

```python
{
    'class_weight': 'balanced',
    'max_depth': None,
    'max_features': 'sqrt',
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 200
}